## Cell 1: Setup
**What this demonstrates**: Environment initialisation for RAG Fusion — Anthropic for query variant generation and answer synthesis, OpenAI for embeddings, Chroma for retrieval.
**Expected output**: Setup confirmation with model names and configuration.

In [ ]:
%pip install -q langchain langchain-openai langchain-community chromadb python-dotenv

import os
import time
import pathlib

from dotenv import load_dotenv, find_dotenv
from anthropic import Anthropic
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv(find_dotenv())
_anthropic_key = os.environ.get('ANTHROPIC_API_KEY', '')
_openai_key    = os.environ.get('OPENAI_API_KEY', '')
assert _anthropic_key, 'ANTHROPIC_API_KEY not set — add it to .env'
assert _openai_key,    'OPENAI_API_KEY not set — add it to .env'

# Haiku generates variants cheaply; Sonnet synthesises the final answer
VARIANT_MODEL = 'claude-haiku-4-5-20251001'
ANSWER_MODEL  = 'claude-sonnet-4-6'
EMBED_MODEL   = 'text-embedding-3-small'
N_VARIANTS    = 3   # number of additional query rephrasings
RRF_K         = 60  # standard RRF smoothing constant
TOP_K         = 5   # chunks retrieved per query variant

client     = Anthropic()
embeddings = OpenAIEmbeddings(model=EMBED_MODEL)

print('Setup complete')
print(f'  Variant model : {VARIANT_MODEL}')
print(f'  Answer model  : {ANSWER_MODEL}')
print(f'  N variants    : {N_VARIANTS}')
print(f'  RRF k         : {RRF_K}')
print(f'  Top-k per run : {TOP_K}')

## Cell 2: Data — Earnings Report
**What this demonstrates**: Loading a multi-aspect earnings report — a document covering revenue, segment performance, guidance, and risk factors. This variety of topics makes it an ideal test case for RAG Fusion: different phrasings of the same question will retrieve different relevant sections.
**Expected output**: Chunk count and a preview of the aspect diversity that RAG Fusion is designed to exploit.

In [ ]:
_base_candidates = [
    pathlib.Path('../../shared/sample_data'),
    pathlib.Path('shared/sample_data'),
]
BASE_DIR = next((p for p in _base_candidates if p.exists()), None)
assert BASE_DIR is not None, 'Cannot find shared/sample_data'

# Earnings report covers multiple aspects:
#   revenue & EPS, segment breakdowns, forward guidance, risk factors
# A single query like "financial highlights" retrieves summary sections.
# Variants targeting "segment performance" or "guidance" surface different chunks.
text = (BASE_DIR / 'earnings_report.txt').read_text(encoding='utf-8')
raw_doc = Document(page_content=text, metadata={'source': 'earnings_report'})

# Small chunks expose the vocabulary diversity across report sections
splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=60)
chunks: list[Document] = splitter.split_documents([raw_doc])

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name='rag_fusion_earnings',
)

print(f'Loaded: {len(text):,} characters → {len(chunks)} chunks')
print()
print('Why aspect diversity matters for RAG Fusion:')
print('  "financial highlights" → retrieves summary / EPS sections')
print('  "quarterly revenue performance" → retrieves income statement sections')
print('  "business segment results" → retrieves segment breakdown sections')
print('  RAG Fusion retrieves all three simultaneously and fuses them')

## Cell 3: Core — Variant Generator, RRF Fusion, RAG Fusion Pipeline
**What this demonstrates**: Three functions implement the full pattern: `generate_query_variants` rephrases the original query, `rrf_fuse` merges ranked retrieval lists, and `rag_fusion` orchestrates the end-to-end pipeline.
**Expected output**: Function definitions confirmed.

In [ ]:
def generate_query_variants(original_query: str, n: int = N_VARIANTS) -> list[str]:
    """Generate n rephrasings of the original query from different angles.

    Each variant preserves the information need but uses different vocabulary
    or framing — exposing document sections that the original phrasing misses.

    Args:
        original_query: The user's question.
        n:              Number of variants to generate.

    Returns:
        List of n rephrased queries (original not included).
    """
    prompt = '\n'.join([
        f'Generate {n} rephrasings of the following query.',
        'Each rephrasing must:',
        '  1. Preserve the original information need exactly',
        '  2. Use different vocabulary, framing, or emphasis',
        '  3. Be a complete, standalone question',
        'Return one rephrasing per line. No numbering, no prefixes.',
        '',
        f'Query: {original_query}',
    ])
    response = client.messages.create(
        model=VARIANT_MODEL,
        max_tokens=300,
        messages=[{'role': 'user', 'content': prompt}],
    )
    lines = [
        line.strip()
        for line in response.content[0].text.strip().splitlines()
        if line.strip()
    ]
    return lines[:n]


def rrf_fuse(
    results_list: list[list[Document]],
    k: int = RRF_K,
) -> list[tuple[Document, float]]:
    """Merge N ranked document lists using Reciprocal Rank Fusion.

    Score formula: score(doc) = sum(1 / (k + rank_i)) across all lists.
    Documents appearing in multiple lists receive additive score boosts.
    The constant k=60 prevents top-ranked documents from dominating.

    Args:
        results_list: N lists of documents, each ranked by a separate retrieval.
        k:            RRF smoothing constant (default 60).

    Returns:
        List of (Document, rrf_score) tuples sorted by descending score.
    """
    scores: dict[str, float]    = {}
    doc_map: dict[str, Document] = {}

    for ranked_list in results_list:
        for rank, doc in enumerate(ranked_list):
            # Use first 80 chars as a stable content key for deduplication
            key = doc.page_content[:80]
            # Additive: same doc in multiple lists accumulates score
            scores[key]  = scores.get(key, 0.0) + 1.0 / (k + rank + 1)
            doc_map[key] = doc

    # Sort descending by fused score
    sorted_keys = sorted(scores, key=lambda kk: scores[kk], reverse=True)
    return [(doc_map[kk], scores[kk]) for kk in sorted_keys]


def rag_fusion(
    query: str,
    n_variants: int = N_VARIANTS,
    top_k: int = TOP_K,
) -> dict:
    """Full RAG Fusion pipeline: generate → retrieve × (1+N) → fuse → generate.

    The original query is always included in the retrieval pool — variants
    expand coverage but never replace the original.

    Args:
        query:      The user's original question.
        n_variants: Number of query rephrasings to generate.
        top_k:      Documents to retrieve per query.

    Returns:
        dict with keys: query, variants, all_queries, all_results,
                        fused, answer, latency_ms.
    """
    t0 = time.perf_counter()

    # Step 1: generate N variants; original always included in pool
    variants    = generate_query_variants(query, n=n_variants)
    all_queries = [query] + variants

    # Step 2: retrieve top-k chunks for each query independently
    all_results: list[list[Document]] = [
        vectorstore.similarity_search(q, k=top_k)
        for q in all_queries
    ]

    # Step 3: fuse all ranked lists — docs in multiple lists score higher
    fused = rrf_fuse(all_results)

    # Step 4: deduplicate — RRF score already boosts multi-list docs;
    #         keep only the first occurrence of each content key
    seen: set[str] = set()
    deduped: list[tuple[Document, float]] = []
    for doc, score in fused:
        key = doc.page_content[:80]
        if key not in seen:
            seen.add(key)
            deduped.append((doc, score))

    # Step 5: build context from top-5 fused chunks, annotated with RRF scores
    context = '\n\n---\n\n'.join(
        f'[rrf={score:.4f}]\n{doc.page_content}'
        for doc, score in deduped[:5]
    )

    # Step 6: generate answer from fused context
    response = client.messages.create(
        model=ANSWER_MODEL,
        max_tokens=400,
        system='Answer using only the provided context. Be specific and cite figures where available.',
        messages=[{'role': 'user', 'content': f'Context:\n{context}\n\nQuestion: {query}'}],
    )

    return {
        'query'       : query,
        'variants'    : variants,
        'all_queries' : all_queries,
        'all_results' : all_results,
        'fused'       : deduped,
        'answer'      : response.content[0].text.strip(),
        'latency_ms'  : (time.perf_counter() - t0) * 1000,
    }


print('RAG Fusion pipeline defined')
print('  generate_query_variants() — N rephrasings via Haiku')
print('  rrf_fuse()                — merge ranked lists via RRF')
print('  rag_fusion()              — generate → retrieve×(1+N) → fuse → generate')

## Cell 4: Run — Q3 Financial Highlights Query
**What this demonstrates**: RAG Fusion on a broad earnings query. Three variants cover different aspects of "financial highlights" (revenue performance, segment results, EPS/guidance); the fused result surfaces chunks that the original single query alone would miss.
**Expected output**: Original query, 3 generated variants, per-variant retrieval results, and the final synthesised answer.

In [ ]:
QUERY = 'What were the key financial highlights in Q3?'

print(f'Query: {QUERY}')
print()

result = rag_fusion(QUERY)

# Show original + generated variants
print('Query variants generated:')
print(f'  [Original]  {result["query"]}')
for i, v in enumerate(result['variants'], 1):
    print(f'  [Variant {i}]  {v}')
print()

# Show top-3 chunks retrieved per query
print('Retrieval results per query (top 3 each):')
print('-' * 65)
for label, docs in zip(result['all_queries'], result['all_results']):
    tag = 'Original' if label == QUERY else 'Variant'
    print(f'\n[{tag}] {label[:60]}')
    for rank, doc in enumerate(docs[:3], 1):
        preview = doc.page_content[:80].replace('\n', ' ')
        print(f'  {rank}. {preview}...')

print()
print('Fused top-5 chunks (after RRF + dedup):')
print('-' * 65)
for rank, (doc, score) in enumerate(result['fused'][:5], 1):
    preview = doc.page_content[:70].replace('\n', ' ')
    print(f'  {rank}. [rrf={score:.4f}] {preview}...')

print()
print('=' * 65)
print('ANSWER')
print('=' * 65)
print(result['answer'])
print()
print(f'Latency: {result["latency_ms"]:.0f} ms  |  '
      f'Queries: {len(result["all_queries"])}  |  '
      f'Fused chunks: {len(result["fused"])}')

## Cell 5: Inspect — RRF Scores, Coverage Analysis, Timing
**What this demonstrates**: The mechanics of fusion — which chunks appeared in multiple variant lists (high RRF score), which were unique to one variant (lower score but still surfaced), and the unique coverage gain of RAG Fusion over a single-query baseline.
**Expected output**: RRF score table, per-variant provenance, and a baseline comparison showing coverage gain.

In [ ]:
# ── RRF score distribution ─────────────────────────────────────────────────────
print('RRF SCORE DISTRIBUTION (top 10 fused chunks)')
print('=' * 70)
print(f"{'Rank':<5} {'RRF score':<12} {'Chunk preview (first 50 chars)'}")
print('-' * 70)
for rank, (doc, score) in enumerate(result['fused'][:10], 1):
    preview = doc.page_content[:50].replace('\n', ' ')
    # Flag chunks that appeared in multiple lists (score > single-list max)
    single_list_max = 1.0 / (RRF_K + 1)          # score if in exactly one list at rank 0
    multi = ' ★ multi-list' if score > single_list_max * 1.5 else ''
    print(f'{rank:<5} {score:<12.5f} {preview}...{multi}')

# ── Per-variant provenance ─────────────────────────────────────────────────────
print()
print('PER-VARIANT PROVENANCE (which list each top-5 chunk came from)')
print('=' * 70)
top5_keys = {doc.page_content[:80] for doc, _ in result['fused'][:5]}
for label, docs in zip(result['all_queries'], result['all_results']):
    tag      = 'Original' if label == QUERY else 'Variant '
    keys     = {d.page_content[:80] for d in docs}
    overlap  = top5_keys & keys
    print(f'  [{tag}] contributed {len(overlap)}/5 top-fused chunks')

# ── Baseline comparison ────────────────────────────────────────────────────────
print()
print('BASELINE COMPARISON: single query vs RAG Fusion top-5')
print('=' * 70)
baseline_docs = vectorstore.similarity_search(QUERY, k=5)
baseline_keys = {d.page_content[:80] for d in baseline_docs}
fused_keys    = {d.page_content[:80] for d, _ in result['fused'][:5]}
new_in_fusion = fused_keys - baseline_keys
lost_in_fusion = baseline_keys - fused_keys

print(f'  Baseline top-5  : {len(baseline_keys)} unique chunks')
print(f'  Fused top-5     : {len(fused_keys)} unique chunks')
print(f'  New in fusion   : {len(new_in_fusion)} chunks baseline missed')
print(f'  Dropped by fusion: {len(lost_in_fusion)} baseline chunks not in fused top-5')
print()

if new_in_fusion:
    print('Chunks caught by RAG Fusion that single-query missed:')
    for key in new_in_fusion:
        print(f'  → {key[:70]}...')
else:
    print('All fused top-5 chunks also appeared in the single-query baseline.')
    print('(Expected on small corpora — effect is stronger at scale)')

# ── Timing breakdown ───────────────────────────────────────────────────────────
print()
print('TIMING BREAKDOWN')
print('=' * 70)
n_queries = len(result['all_queries'])
print(f'  Total latency   : {result["latency_ms"]:.0f} ms')
print(f'  Queries run     : {n_queries} (1 original + {n_queries - 1} variants)')
print(f'  Est. retrieval  : ~{n_queries * TOP_K} vector ops ({n_queries} × {TOP_K})')
print(f'  Note: parallelise retrievals with asyncio.gather to cut latency by ~{n_queries}×')

## Cell 6: Fintech — Market Risk Multi-Angle Analysis
**What this demonstrates**: RAG Fusion on a market risk query where different variants naturally target different risk categories in the earnings report (credit risk, liquidity risk, market risk, operational risk). This multi-angle retrieval captures the complete risk picture that a single query would underrepresent.
**Expected output**: Risk-themed variants, fused results spanning multiple risk factor sections, and an answer covering more risk dimensions than single-query retrieval.

In [ ]:
RISK_QUERY = 'What are the market risks mentioned?'

print('FINTECH: MARKET RISK — MULTI-ANGLE ANALYSIS')
print(f'Query: {RISK_QUERY}')
print()
print('Why this benefits from RAG Fusion:')
print('  Earnings reports cover risk under many headings:')
print('  "credit risk", "liquidity risk", "interest rate sensitivity",')
print('  "counterparty exposure", "regulatory capital risk"')
print('  A single query on "market risks" retrieves one cluster.')
print('  Variants targeting each category surface the full picture.')
print()

risk_result = rag_fusion(RISK_QUERY)

# Show variants with their implied risk angle
print('Generated variants (each targets a different risk angle):')
print(f'  [Original]  {risk_result["query"]}')
for i, v in enumerate(risk_result['variants'], 1):
    print(f'  [Variant {i}]  {v}')
print()

# Show fused results with RRF scores
print('Fused top-5 risk chunks:')
print('-' * 65)
for rank, (doc, score) in enumerate(risk_result['fused'][:5], 1):
    preview = doc.page_content[:100].replace('\n', ' ')
    print(f'{rank}. [rrf={score:.4f}] {preview}...')
    print()

# Baseline comparison for the risk query
baseline_risk = vectorstore.similarity_search(RISK_QUERY, k=5)
baseline_risk_keys = {d.page_content[:80] for d in baseline_risk}
fused_risk_keys    = {d.page_content[:80] for d, _ in risk_result['fused'][:5]}
new_risk_chunks    = fused_risk_keys - baseline_risk_keys

print(f'Coverage gain: {len(new_risk_chunks)} additional risk chunks vs single-query baseline')
print()
print('=' * 65)
print('RISK ANALYSIS ANSWER')
print('=' * 65)
print(risk_result['answer'])
print()
print('-' * 65)
print('RAG Fusion value for risk analysis:')
print('  Each variant targets a different risk terminology cluster')
print('  RRF surfaces risk factors that appear across multiple sections')
print('  Result: a more complete risk picture than any single-angle query')